# IBM Employee Attrition - Interview Walkthrough

This notebook is structured for a live interview narrative:

1. Problem framing
2. Data and quick EDA
3. Model performance and subgroup checks
4. Actionable business recommendations

## Business Context

Employee attrition is expensive due to replacement cost, lost productivity, and team disruption.
We frame this as a binary classification problem where `Attrition = 1` indicates likely churn.

In [1]:
from pathlib import Path
import json
import subprocess

import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

project_root = Path.cwd().resolve().parent
artifacts_dir = project_root / "artifacts"
processed_data_path = project_root / "data" / "processed" / "synthetic_attrition.csv"
metrics_path = artifacts_dir / "metrics.json"
feature_importance_path = artifacts_dir / "feature_importance.csv"
predictions_path = artifacts_dir / "predictions.csv"

print("Project root:", project_root)

Project root: /Users/danielharrod/Data_Science/IBM_Employee_Attrition_ML


In [2]:
# Ensure training artifacts exist so the notebook is runnable end-to-end.
if not metrics_path.exists() or not predictions_path.exists():
    print("Artifacts not found. Running training script...")
    subprocess.run(["python3", str(project_root / "src" / "train.py")], check=True)

with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

raw_data = pd.read_csv(processed_data_path)
predictions = pd.read_csv(predictions_path)
feature_importance = pd.read_csv(feature_importance_path)

print("Rows in dataset:", len(raw_data))
print("Rows in test predictions:", len(predictions))
print("Selected model:", metrics["selected_model"])

Rows in dataset: 2000
Rows in test predictions: 400
Selected model: logistic_regression


## 1) Quick EDA

In interviews, keep EDA concise and hypothesis-driven:

- Is target balanced?
- Which workforce segments show higher attrition?
- Do high-friction signals (e.g., overtime, commute) align with risk?

In [3]:
attrition_rate = raw_data["Attrition"].mean()
print(f"Overall attrition rate: {attrition_rate:.2%}")

summary_cols = ["OverTime", "Department", "MaritalStatus", "Gender"]
for col in summary_cols:
    segment = (
        raw_data.groupby(col, dropna=False)["Attrition"]
        .agg(["count", "mean"])
        .rename(columns={"count": "n", "mean": "attrition_rate"})
        .sort_values("attrition_rate", ascending=False)
    )
    print(f"\nAttrition by {col}:")
    display(segment)

corr_snapshot = raw_data[["Attrition", "Age", "MonthlyIncome", "YearsAtCompany", "DistanceFromHome", "JobSatisfaction", "WorkLifeBalance"]].corr(numeric_only=True)
display(corr_snapshot[["Attrition"]].sort_values("Attrition", ascending=False))

Overall attrition rate: 1.00%

Attrition by OverTime:


,n,attrition_rate
OverTime,,
Yes,550,0.021818
No,1450,0.005517



Attrition by Department:


,n,attrition_rate
Department,,
Research & Development,1096,0.012774
Sales,696,0.007184
Human Resources,208,0.004808



Attrition by MaritalStatus:


,n,attrition_rate
MaritalStatus,,
Single,712,0.011236
Married,959,0.010428
Divorced,329,0.006079



Attrition by Gender:


,n,attrition_rate
Gender,,
Female,988,0.011134
Male,1012,0.008893


,Attrition
Attrition,1.000000
DistanceFromHome,0.024485
WorkLifeBalance,-0.024876
MonthlyIncome,-0.027827
Age,-0.043446
YearsAtCompany,-0.063382
JobSatisfaction,-0.070068


## 2) Model Selection and Quality

The training pipeline compares logistic regression and random forest using cross-validated ROC-AUC, then evaluates the best model on a holdout test set.

For interview discussion, emphasize both:

- **Discrimination quality** (ROC-AUC)
- **Decision quality at threshold** (precision/recall/F1)

In [4]:
print("Cross-validated ROC-AUC by model:")
for model_name, auc in metrics["cv_roc_auc"].items():
    print(f"  {model_name}: {auc:.3f}")

print("\nSelected model:", metrics["selected_model"])
print("\nTest metrics:")
for metric_name, value in metrics["test_metrics"].items():
    print(f"  {metric_name}: {value:.3f}")

print("\nSubgroup metrics (precision/recall):")
for group_name, group_values in metrics["subgroup_metrics"].items():
    print(f"\n{group_name}")
    if not group_values:
        print("  (no eligible segments)")
        continue
    display(pd.DataFrame(group_values).T.sort_values("n_samples", ascending=False))

Cross-validated ROC-AUC by model:
  logistic_regression: 0.730
  random_forest: 0.627

Selected model: logistic_regression

Test metrics:
  accuracy: 0.990
  precision: 0.000
  recall: 0.000
  f1: 0.000
  roc_auc: 0.870

Subgroup metrics (precision/recall):

Gender


,n_samples,precision,recall
Male,201.0,0.0,0.0
Female,199.0,0.0,0.0



OverTime


,n_samples,precision,recall
No,288.0,0.0,0.0
Yes,112.0,0.0,0.0


## 3) Explainability + Threshold Strategy

Interviewers often ask: *"How would you use this model in practice?"*

Two practical levers:

1. **Feature importance** to explain key drivers.
2. **Threshold tuning** to align with business trade-offs (e.g., fewer misses vs fewer false alarms).

In [5]:
print("Top 15 model drivers:")
display(feature_importance.head(15))

y_true = predictions["actual_attrition"].astype(int)
y_prob = predictions["predicted_attrition_prob"].astype(float)

threshold_grid = np.arange(0.30, 0.76, 0.05)
rows = []
for threshold in threshold_grid:
    y_pred = (y_prob >= threshold).astype(int)
    rows.append(
        {
            "threshold": round(float(threshold), 2),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "predicted_positive_rate": float(y_pred.mean()),
        }
    )

threshold_table = pd.DataFrame(rows)
display(threshold_table.style.format({"precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}", "predicted_positive_rate": "{:.3f}"}))

Top 15 model drivers:


,feature,importance
0,cat__OverTime_No,0.709163
1,num__JobSatisfaction,0.694966
2,cat__OverTime_Yes,0.693724
3,num__YearsAtCompany,0.590765
4,num__Age,0.570887
5,cat__JobRole_Human Resources,0.501382
6,cat__Department_Research & Development,0.442411
7,cat__Department_Human Resources,0.311636
8,cat__JobRole_Sales Executive,0.259837
9,cat__MaritalStatus_Divorced,0.221622


,threshold,precision,recall,f1,predicted_positive_rate
0,0.300000,0.000,0.000,0.000,0.000
1,0.350000,0.000,0.000,0.000,0.000
2,0.400000,0.000,0.000,0.000,0.000
3,0.450000,0.000,0.000,0.000,0.000
4,0.500000,0.000,0.000,0.000,0.000
5,0.550000,0.000,0.000,0.000,0.000
6,0.600000,0.000,0.000,0.000,0.000
7,0.650000,0.000,0.000,0.000,0.000
8,0.700000,0.000,0.000,0.000,0.000
9,0.750000,0.000,0.000,0.000,0.000


## 4) IBM-Style Recommendations

If this model were deployed in HR analytics, a practical plan would be:

- Use monthly batch scoring to produce a ranked retention-risk queue.
- Start with a conservative threshold to prioritize high-confidence interventions.
- Route high-risk employees to tailored actions (manager check-ins, work-life support, compensation review, commute flexibility).
- Monitor precision/recall drift and subgroup performance every month.
- Retrain quarterly (or earlier if drift exceeds a policy threshold).

## Interview Closing Statement

"I built this as an end-to-end pipeline: clear business framing, reproducible preprocessing, model comparison, holdout validation, subgroup checks, and deployment-ready artifacts. My next production step would be model monitoring and intervention impact measurement."